<a href="https://colab.research.google.com/github/f3r21/actas-cnn/blob/main/notebooks/03_ablaciones_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# actas-cnn — Ablacion ink-aware (Colab)

Re-entrena las **3 variantes** (base, ls_ra, ls_ra_mu_cos) sobre los crops **ink-aware** publicados por `01_preprocesamiento_colab.ipynb` y las compara en **val y test** en una sola tabla, sin mezclar etiquetados (right-justified vs ink-aware). `ls_ra_mu_cos` es la misma receta que entrena el entregable `02`.

**Prerequisito:** `01_preprocesamiento_colab.ipynb` ya publico el bundle ink-aware en HF.

**Como correr:** Runtime -> Change runtime type -> **T4 GPU**, luego Run all. Son 3 entrenamientos: ~20-35 min en T4.

## 0. Setup

In [1]:
# Dependencias (Colab ya trae torch, torchvision, numpy, pandas, matplotlib)
%pip install -q pymupdf==1.27.2.3 opencv-python-headless huggingface_hub pyarrow
print("deps instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 79.3 MB/s eta 0:00:00
deps instaladas


In [2]:
# === Config + entorno ===
import os, tarfile
from pathlib import Path
import pandas as pd
import torch

def torch_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
DEVICE = torch_device(); print("device:", DEVICE)
if DEVICE.type != "cuda":
    print("AVISO: sin GPU CUDA. En Colab: Runtime -> Change runtime type -> T4 GPU. "
          "Sin GPU el entrenamiento es MUY lento.")

HF_DATASET_REPO = "f3r21/actas-cnn-dataset"   # PDFs + labels (+ crops_bundle si 01 ya corrio)

WORK = Path("/content") if Path("/content").exists() else Path(".").resolve()
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
# Epochs por variante (~5-8 min cada una en T4).
EPOCHS = 20

device: cuda


In [3]:
# Plantilla Presidencial: 42 campos (38 partidos + blanco/nulos/impugnados + total).
# Cajas en fraccion [0,1]; embebida para que el notebook sea autonomo.
TEMPLATE = { "descripcion": "ACTA DE ESCRUTINIO PRESIDENCIAL (idEleccion=10, tipo=1). iter7 auto-detected.", "image_size_reference": [ 2339, 3309 ], "fields": [ { "name": "partido_01", "box": [ 0.462, 0.2149, 0.539, 0.2315 ], "n_digits": 3 }, { "name": "partido_02", "box": [ 0.462, 0.2315, 0.539, 0.2481 ], "n_digits": 3 }, { "name": "partido_03", "box": [ 0.462, 0.2481, 0.539, 0.2648 ], "n_digits": 3 }, { "name": "partido_04", "box": [ 0.462, 0.2648, 0.539, 0.2814 ], "n_digits": 3 }, { "name": "partido_05", "box": [ 0.462, 0.2814, 0.539, 0.298 ], "n_digits": 3 }, { "name": "partido_06", "box": [ 0.462, 0.298, 0.539, 0.3146 ], "n_digits": 3 }, { "name": "partido_07", "box": [ 0.462, 0.3146, 0.539, 0.3312 ], "n_digits": 3 }, { "name": "partido_08", "box": [ 0.462, 0.3312, 0.539, 0.3479 ], "n_digits": 3 }, { "name": "partido_09", "box": [ 0.462, 0.3479, 0.539, 0.3645 ], "n_digits": 3 }, { "name": "partido_10", "box": [ 0.462, 0.3645, 0.539, 0.3811 ], "n_digits": 3 }, { "name": "partido_11", "box": [ 0.462, 0.3811, 0.539, 0.3977 ], "n_digits": 3 }, { "name": "partido_12", "box": [ 0.462, 0.3977, 0.539, 0.4143 ], "n_digits": 3 }, { "name": "partido_13", "box": [ 0.462, 0.4143, 0.539, 0.431 ], "n_digits": 3 }, { "name": "partido_14", "box": [ 0.462, 0.431, 0.539, 0.4476 ], "n_digits": 3 }, { "name": "partido_15", "box": [ 0.462, 0.4476, 0.539, 0.4642 ], "n_digits": 3 }, { "name": "partido_16", "box": [ 0.462, 0.4642, 0.539, 0.4808 ], "n_digits": 3 }, { "name": "partido_17", "box": [ 0.462, 0.4808, 0.539, 0.4974 ], "n_digits": 3 }, { "name": "partido_18", "box": [ 0.462, 0.4974, 0.539, 0.5141 ], "n_digits": 3 }, { "name": "partido_19", "box": [ 0.462, 0.5141, 0.539, 0.5307 ], "n_digits": 3 }, { "name": "partido_20", "box": [ 0.462, 0.5307, 0.539, 0.5473 ], "n_digits": 3 }, { "name": "partido_21", "box": [ 0.462, 0.5473, 0.539, 0.5639 ], "n_digits": 3 }, { "name": "partido_22", "box": [ 0.462, 0.5639, 0.539, 0.5805 ], "n_digits": 3 }, { "name": "partido_23", "box": [ 0.462, 0.5805, 0.539, 0.5972 ], "n_digits": 3 }, { "name": "partido_24", "box": [ 0.462, 0.5972, 0.539, 0.6138 ], "n_digits": 3 }, { "name": "partido_25", "box": [ 0.462, 0.6138, 0.539, 0.6304 ], "n_digits": 3 }, { "name": "partido_26", "box": [ 0.462, 0.6304, 0.539, 0.647 ], "n_digits": 3 }, { "name": "partido_27", "box": [ 0.462, 0.647, 0.539, 0.6636 ], "n_digits": 3 }, { "name": "partido_28", "box": [ 0.462, 0.6636, 0.539, 0.6803 ], "n_digits": 3 }, { "name": "partido_29", "box": [ 0.462, 0.6803, 0.539, 0.6969 ], "n_digits": 3 }, { "name": "partido_30", "box": [ 0.462, 0.6969, 0.539, 0.7135 ], "n_digits": 3 }, { "name": "partido_31", "box": [ 0.462, 0.7135, 0.539, 0.7301 ], "n_digits": 3 }, { "name": "partido_32", "box": [ 0.462, 0.7301, 0.539, 0.7467 ], "n_digits": 3 }, { "name": "partido_33", "box": [ 0.462, 0.7467, 0.539, 0.7634 ], "n_digits": 3 }, { "name": "partido_34", "box": [ 0.462, 0.7634, 0.539, 0.78 ], "n_digits": 3 }, { "name": "partido_35", "box": [ 0.462, 0.78, 0.539, 0.7966 ], "n_digits": 3 }, { "name": "partido_36", "box": [ 0.462, 0.7966, 0.539, 0.8132 ], "n_digits": 3 }, { "name": "partido_37", "box": [ 0.462, 0.8132, 0.539, 0.8298 ], "n_digits": 3 }, { "name": "partido_38", "box": [ 0.462, 0.8298, 0.539, 0.8465 ], "n_digits": 3 }, { "name": "votos_blanco", "box": [ 0.462, 0.8465, 0.539, 0.8631 ], "n_digits": 3 }, { "name": "votos_nulos", "box": [ 0.462, 0.8631, 0.539, 0.8797 ], "n_digits": 3 }, { "name": "votos_impugnados", "box": [ 0.462, 0.8797, 0.539, 0.8963 ], "n_digits": 3 }, { "name": "total_ciudadanos", "box": [ 0.4363, 0.8963, 0.539, 0.913 ], "n_digits": 4 } ] }

## 1. Codigo del modelo (inline)

In [4]:
# === Modelo del proyecto: ResNet-18 estilo CIFAR (He et al., 2015) ===
import torch.nn as nn
from torchvision.models import resnet18 as _torchvision_resnet18

def resnet18_cifar(in_channels=1, num_classes=10):
    """ResNet-18 adaptada a entradas 1x32x32: stem 3x3 stride 1, sin MaxPool
    inicial (preserva resolucion), 1 canal. Mantiene las 4 etapas residuales,
    GAP (1,1) y Linear(512, num_classes). 11.17M params."""
    m = _torchvision_resnet18(num_classes=num_classes)
    m.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    return m

In [5]:
# === Dataset de crops + transforms ===
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

def default_transforms(image_size=32, train=True, randaugment=False):
    ops = [transforms.Grayscale(num_output_channels=1),
           transforms.Resize((image_size, image_size))]
    if train:
        ops.append(transforms.RandomAffine(degrees=8, translate=(0.1, 0.1), scale=(0.9, 1.1)))
        if randaugment:
            ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
    ops += [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
    return transforms.Compose(ops)

class CropsDataset(Dataset):
    def __init__(self, manifest_csv, root=".", transform=None):
        self.df = pd.read_csv(manifest_csv)
        self.root = Path(root)
        self.transform = transform or default_transforms()
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = self.transform(Image.open(self.root / row["path"]))
        return x, torch.tensor(int(row["label"]), dtype=torch.long)

In [6]:
# === Entrenamiento (ResNet-18, recipe base, ~5-8 min en T4) ===
import os
import numpy as np
from torch.utils.data import DataLoader, random_split

def _mixup(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def run_epoch(model, loader, device, criterion, optimizer=None, mixup_alpha=0.0):
    is_train = optimizer is not None
    model.train(is_train)
    total = correct = 0; loss_sum = 0.0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train and mixup_alpha > 0:
                xm, ya, yb, lam = _mixup(x, y, mixup_alpha)
                out = model(xm)
                loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
                correct += (out.argmax(1) == ya).sum().item()
            else:
                out = model(x); loss = criterion(out, y)
                correct += (out.argmax(1) == y).sum().item()
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * x.size(0); total += x.size(0)
    return loss_sum / total, correct / total

def train_model(manifest, root, device, epochs=20, lr=5e-4, batch_size=128,
                label_smoothing=0.0, randaugment=False, mixup=0.0, cosine_lr=False,
                seed=42):
    # Defaults = recipe BASE (sin RandAugment): rapido y consistente con el
    # checkpoint oficial resnet18_best.pt (~90.3% acta). RandAugment corre en CPU
    # por imagen y es el cuello de botella en Colab; activalo (randaugment=True,
    # mixup=0.2, cosine_lr=True, label_smoothing=0.1) solo para la ablacion ls_ra_mu_cos.
    torch.manual_seed(seed)
    full = CropsDataset(manifest, root=root,
                        transform=default_transforms(32, train=True, randaugment=randaugment))
    n_val = max(1, int(0.2 * len(full)))
    tr, va = random_split(full, [len(full) - n_val, n_val])
    pin = device.type == "cuda"
    nw = min(4, os.cpu_count() or 2)
    trl = DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=nw,
                     pin_memory=pin, persistent_workers=nw > 0)
    val = DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=nw,
                     pin_memory=pin, persistent_workers=nw > 0)
    model = resnet18_cifar(1, 10).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine_lr else None
    best = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        _, tra = run_epoch(model, trl, device, crit, opt, mixup_alpha=mixup)
        _, vaa = run_epoch(model, val, device, crit)
        if sched: sched.step()
        if vaa > best:
            best = vaa
            import copy
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, "resnet18_best.pt")
        print(f"epoch {ep:02d}  train_acc {tra:.4f}  val_acc {vaa:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"mejor val_acc (holdout interno): {best:.4f} guardado en resnet18_best.pt")
    return model

In [7]:
# === Evaluacion downstream: digit / field / acta-level + reconstruccion del total ===
import numpy as np

def field_value_for(name, votos_acta, total_emitidos):
    """Entero del ground truth para un campo (partido / blanco-nulos-impugnados / total)."""
    if name.startswith("partido_"):
        pos = int(name.split("_")[1])
        row = votos_acta[votos_acta["nposicion"] == pos]
        return int(row.iloc[0]["nvotos"]) if len(row) else 0
    mapping = {"votos_blanco": 80, "votos_nulos": 81, "votos_impugnados": 82}
    if name in mapping:
        row = votos_acta[votos_acta["nposicion"] == mapping[name]]
        return int(row.iloc[0]["nvotos"]) if len(row) else 0
    if name == "total_ciudadanos":
        return int(total_emitidos)
    raise ValueError(name)

def parse_crop_path(rel):
    """'<label>/<aid>_<field>_c<pos>.png' -> (aid, field, pos)."""
    parts = Path(rel).stem.split("_")
    return parts[0], "_".join(parts[1:-1]), int(parts[-1][1:])

def reconstruct_value(preds_by_pos):
    """Concatena los digitos predichos de las celdas presentes, en orden.
    Right-justified e ink-aware por igual: la posicion es orden de lectura."""
    if not preds_by_pos:
        return 0
    return int("".join(str(preds_by_pos[p]) for p in sorted(preds_by_pos)))

@torch.no_grad()
def evaluate_split(model, manifest, crops_root, template, archivos, votos, cabecera, device):
    """Devuelve (df_celdas, res_campos): digit-level en df, field/acta-level en res."""
    model.eval()
    field_specs = {f["name"]: f["n_digits"] for f in template["fields"]}
    aid_to_idacta = dict(zip(archivos["archivoId"], archivos["idActa"]))
    ds = CropsDataset(manifest, root=crops_root, transform=default_transforms(32, train=False))
    df = ds.df.reset_index(drop=True)
    preds = []
    loader = DataLoader(ds, batch_size=512, shuffle=False,
                        num_workers=min(4, os.cpu_count() or 2))
    for x, _ in loader:
        preds.append(model(x.to(device)).argmax(1).cpu().numpy())
    df["pred"] = np.concatenate(preds)
    parsed = df["path"].apply(parse_crop_path).apply(pd.Series)
    parsed.columns = ["archivoId", "field", "pos"]
    df = pd.concat([df, parsed], axis=1)
    rows = []
    for aid, da in df.groupby("archivoId"):
        if aid not in aid_to_idacta:
            continue
        ida = int(aid_to_idacta[aid])
        cab = cabecera[cabecera["idActa"] == ida]
        if len(cab) == 0 or pd.isna(cab.iloc[0]["totalVotosEmitidos"]):
            continue
        total = int(cab.iloc[0]["totalVotosEmitidos"])
        va = votos[votos["idActa"] == ida]
        for fname in field_specs:
            cf = da[da["field"] == fname]
            pv = reconstruct_value(dict(zip(cf["pos"], cf["pred"])))
            rv = field_value_for(fname, va, total)
            rows.append({"archivoId": aid, "field": fname, "pred": pv, "real": rv,
                         "correct": pv == rv, "error": pv - rv})
    res = pd.DataFrame(rows)
    n_eval, n_total = res["archivoId"].nunique(), df["archivoId"].nunique()
    if n_eval < n_total:
        print(f"aviso: {n_eval}/{n_total} actas evaluadas "
              f"({n_total - n_eval} sin ground truth en los parquets)")
    return df, res

def report_metrics(df, res, split="val"):
    """Imprime y devuelve el dict de metricas oficiales."""
    digit = float(df["pred"].eq(df["label"]).mean())
    field = float(res["correct"].mean())
    acta = float(res.groupby("archivoId")["correct"].all().mean())
    no_tot = res[res["field"] != "total_ciudadanos"]
    err = (no_tot.groupby("archivoId")["pred"].sum() - no_tot.groupby("archivoId")["real"].sum())
    mae = float(err.abs().mean()); exact = float((err == 0).mean() * 100)
    print(f"== {split.upper()} ==")
    print(f"digit-level : {digit:.4f}  (n={len(df)})")
    print(f"field-level : {field:.4f}")
    print(f"acta-level  : {acta:.4f}")
    print(f"reconstruccion total: MAE {mae:.2f} votos, exacta {exact:.2f}% de actas")
    return {"digit": digit, "field": field, "acta": acta, "total_mae": mae,
            "total_exact_pct": exact}

## 2. Datos: crops ink-aware + labels

In [8]:
from huggingface_hub import hf_hub_download, snapshot_download

snapshot_download(HF_DATASET_REPO, repo_type="dataset", allow_patterns="labels/*",
                  local_dir=str(DATA))
archivos = pd.read_parquet(DATA / "labels/actas_archivos.parquet")
votos    = pd.read_parquet(DATA / "labels/actas_votos.parquet")
cabecera = pd.read_parquet(DATA / "labels/actas_cabecera.parquet")

# Crops preprocesados publicados por 01_preprocesamiento_colab.ipynb.
try:
    bundle = hf_hub_download(HF_DATASET_REPO, "crops_bundle.tar.gz", repo_type="dataset")
except Exception as e:
    raise RuntimeError("No hay crops_bundle.tar.gz en HF. Corre "
                       "01_preprocesamiento_colab.ipynb primero.") from e
with tarfile.open(bundle) as t: t.extractall(WORK)
print("crops_bundle extraido")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

crops_bundle.tar.gz:   0%|          | 0.00/346M [00:00<?, ?B/s]

/tmp/ipykernel_628/2314762994.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(bundle) as t: t.extractall(WORK)


crops_bundle extraido


## 3. Ablacion: entrena 3 variantes y compara (val + test)

Cada variante se entrena desde cero con su receta y se evalua en val y test (reconstruccion vs parquets ONPE). La tabla va a `data/ablations_ink_summary.csv`; los checkpoints `resnet18_ink_*_best.pt` quedan en la VM (efimeros) — subilos a HF si queres conservar el ganador.

In [ ]:
# Ablacion ink-aware: re-entrena 3 variantes sobre los crops ink-aware y compara.
# ls_ra_mu_cos es la receta del entregable (02); base y ls_ra son los escalones previos.
import pandas as pd

VARIANTES = [
    ("base",         {}),
    ("ls_ra",        {"label_smoothing": 0.1, "randaugment": True}),
    ("ls_ra_mu_cos", {"label_smoothing": 0.1, "randaugment": True,
                      "mixup": 0.2, "cosine_lr": True}),
]

filas = []
for nombre, kw in VARIANTES:
    print(f"\n========== entrenando {nombre} ==========")
    m = train_model(DATA / "manifest_train.csv", DATA / "crops_train", DEVICE,
                    epochs=EPOCHS, **kw)
    torch.save({"model": m.state_dict(), "arch": "resnet18"},
               WORK / f"resnet18_ink_{nombre}_best.pt")
    dfv, resv = evaluate_split(m, DATA / "manifest_val.csv", DATA / "crops_val",
                               TEMPLATE, archivos, votos, cabecera, DEVICE)
    mv = report_metrics(dfv, resv, split=f"val/{nombre}")
    dft, rest = evaluate_split(m, DATA / "manifest_test.csv", DATA / "crops_test",
                               TEMPLATE, archivos, votos, cabecera, DEVICE)
    mt = report_metrics(dft, rest, split=f"test/{nombre}")
    filas.append({"variante": nombre,
                  "val_digit": mv["digit"], "val_field": mv["field"],
                  "val_acta": mv["acta"], "val_mae": mv["total_mae"],
                  "test_digit": mt["digit"], "test_field": mt["field"],
                  "test_acta": mt["acta"], "test_mae": mt["total_mae"]})

tabla = pd.DataFrame(filas)
print("\n=== Ablacion ink-aware (val + test) ===")
print(tabla.round(4).to_string(index=False))
tabla.to_csv(DATA / "ablations_ink_summary.csv", index=False)
print("\nguardado: data/ablations_ink_summary.csv + checkpoints resnet18_ink_*_best.pt (en la VM, efimeros)")


========== entrenando base ==========
epoch 01  train_acc 0.9216  val_acc 0.9670
epoch 02  train_acc 0.9755  val_acc 0.9763
epoch 03  train_acc 0.9785  val_acc 0.9723
epoch 04  train_acc 0.9792  val_acc 0.9809
epoch 05  train_acc 0.9808  val_acc 0.9733
epoch 06  train_acc 0.9812  val_acc 0.9769
epoch 07  train_acc 0.9818  val_acc 0.9812
epoch 08  train_acc 0.9827  val_acc 0.9811
epoch 09  train_acc 0.9832  val_acc 0.9819
epoch 10  train_acc 0.9836  val_acc 0.9817
epoch 11  train_acc 0.9839  val_acc 0.9822
epoch 12  train_acc 0.9840  val_acc 0.9808
epoch 13  train_acc 0.9840  val_acc 0.9824
epoch 14  train_acc 0.9846  val_acc 0.9835
epoch 15  train_acc 0.9854  val_acc 0.9839
epoch 16  train_acc 0.9851  val_acc 0.9837
epoch 17  train_acc 0.9853  val_acc 0.9806
epoch 18  train_acc 0.9857  val_acc 0.9800
epoch 19  train_acc 0.9860  val_acc 0.9828
epoch 20  train_acc 0.9863  val_acc 0.9844
mejor val_acc (holdout interno): 0.9844 guardado en resnet18_best.pt
== VAL/BASE ==
digit-level : 0.9

---
Listo. Tabla comparativa ink-aware (val+test). Para promover el ganador a oficial, sube su `resnet18_ink_<variante>_best.pt` a `f3r21/actas-cnn-model` y actualiza README/CLAUDE.